---
authors:
  - edesz
date: 2025-05-02
---

# Validation

## About

This step performs cost-sensitive Machine Learning (ML) model development using tree-based and linear models.

### ML Experiment Inputs

In order to account for the strong class imbalance, nested cross-validation (CV) is performed using `scikit-learn`'s `TunedThresholdClassifierCV()` within `cross_validate()` on the combined training and validation data.

This gives the best classifier decision threshold (via inner CV).

Outer CV is then used to compare the choices of

1. classifier (i.e. the ML model)
2. features
3. feature pre-processing (based on the selected features)

This approach follows the example in the [`scikit-learn` documentation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_tuned_decision_threshold.html#tuning-the-decision-threshold).

In order to prevent data leakage, these three choices are combined into a single `scikit-learn` `Pipeline`, which is passed to the nested cross-validation function. The choices are compared using ML experiments. Details are explained later.

The choice of features is specifically to [test the hypothesis we formed in the EDA step](`./03_eda_v2.ipynb`) that only numerical features are sufficient to predict credict card customer churn. To verify this hypothesis, we will run experiments that include or exclude the other types of features (categorical and ordinal). The scores from these experiments can be used to prove or disprove the hypothesis: if scores do not improve with the other two types of features then we have proved the hypothesis that only numerical features in the sample credit card customer data are sufficient to predict churn.

The out-of-sample performance of the best combination of the three choices above is evaluated against the test data, which was not used in model development.

### Workflow

This step is used to run a single validation experiment. The next step compares all runs of this step (i.e. all ML experiments) to determine the best choices for each of the three inputs from above.

The [Metaflow](https://pypi.org/project/metaflow/) framework is used to define a paramterized flow that can be run with parameters that are combinations of the three choices from above. `feat_group` and `model_fpath` are the two flow parameters that change from one flow run to the next. `experiment_num` is an experiment run counter to keep track of the experiment runs. Other flow parameters do not change across runs.

`feat_group` is specified directly as a string. It controls the choice of features. Feature pre-prcoessing is set internally by the flow and changes based on the choice of features specified in `feat_group`. Two variables specified in the **User Inputs** section (`experiment_num` and `clf_name`) determine the value of `model_fpath`. Based on these choices, this step can be used to run one experiment at a time. In this way, by changing the three variables `feat_group`, `experiment_num` and `clf_name`, the parameters of each Metaflow run are changed.

:::{note}
### Outputs

Artifacts of each Metaflow validation flow run are stored localy in the `notebooks/.metaflow/ValidationFlow/` directory. Nothing is exported to the R2 bucket.
:::

## Python Imports

In [ ]:
import os
from pathlib import Path

import sklearn.ensemble as skens
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.linear_model import LogisticRegression

`joblib` is imported below since it is needed to serialize untrained model objects to the local disk, for use in the custom Metaflow flow for validation. Metaflow is the ML experimentation framework used here. So, `joblib` is *also* imported later in the notebook cell that is used to both define the Metaflow flow, which [requires all necessary imports in the same notebook cell, and then to run it](https://docs.metaflow.org/metaflow/managing-flows/notebook-runs). Similarly, `pathlib` is imported here to speficy the filepath save the models to. It is *also* imported later in the cell that runs the Metaflow to help extract the name of the model from its filepath.

## Python Imports

The required Python modules are imported below

In [ ]:
from pathlib import Path

import joblib

Define the path to the project root directory

In [ ]:
PROJ_ROOT = Path.cwd().parent

Load environment variables with secrets for use in `boto3`

In [ ]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

The required custom modules (not for use in the validation flow itself) are imported below

In [ ]:
from utils.display_utils import pygments_highlight

## User Inputs

Below we define variables that will be used later to run the Metaflow validation flow

In [ ]:
# R2 data bucket details
prefix = "cloud-run"
r2_key_train = f"{prefix}/train_data.parquet.gzip"
r2_key_val = f"{prefix}/validation_data.parquet.gzip"

# datatypes for categorical and ordinal columns
dtypes_ordinals = {
    "income_category": "string[pyarrow]",
    "education_level": "string[pyarrow]",
}
dtypes_categoricals = {
    "gender": "string[pyarrow]",
    "marital_status": "string[pyarrow]",
    "card_category": "string[pyarrow]",
}

ordinal_features = ["income_category", "education_level"]
categorical_features = ["gender", "marital_status", "dependent_count"]

# keep
# - 'credit_limit' and exclude "avg_open_to_buy" which is correlated
# - 'total_revolv_bal' and exclude "avg_utilization_ratio" which is correlated
# - 'total_trans_amt' and exclude 'total_trans_ct'
numeric_features_1 = [
    "months_on_book",
    "num_products",
    "months_inactive_12_mon",
    "contacts_count_12_mon",
    "total_amt_chng_q4_q1",
    "total_trans_amt",
    "total_ct_chng_q4_q1",
    "credit_limit",
    "total_revolv_bal",
]

# keep
# - 'avg_open_to_buy' and exclude "credit_limit" which is correlated
# - 'avg_utilization_ratio' and exclude "total_revolv_bal" which is correlated
# - 'total_trans_ct' and exclude 'total_trans_amt' which is correlated
numeric_features_2 = [
    "months_on_book",
    "num_products",
    "months_inactive_12_mon",
    "contacts_count_12_mon",
    "total_amt_chng_q4_q1",
    "total_trans_ct",
    "total_ct_chng_q4_q1",
    "avg_open_to_buy",
    "avg_utilization_ratio",
]

# variable metaflow experiment run parameters
feat_group = "[numericals_1]"
experiment_num = 1
clf_name = "LogisticRegression"

```{attention}
Every parameter from the above cell is also a parameter of the Metaflow validation flow.

The following parameters are changed from one validation experiment run to another

1. `feat_group`
2. `experiment_num`
3. `clf_name`

All other parameters do not change from one run of the flow to another.
```

Below is the path to the ML models directory in which untrained `scikit-learn` and `xgboost` models will be stored

In [ ]:
models_dir = PROJ_ROOT / "models"

The filepath of a single model will be passed to the validation below so it can be loaded when the flow is run.

## Validation

### Approach to Handle Class Imbalance

Next, we will run experiments to find the best performing ML pipeline.

A pipeline will consist of the following steps

1. feature transformation
   - grouping of infrequent categories
2. feature pre-processing
   - numerical features
     - min-max scaling
   - ordinal features
     - ordinal encoding
   - categorical features
     - one-hot encoding, or
     - no encoding
3. classification model (classifier)
   - `scikit-learn` models (`LogisticRegression`, `RandomForestClassifier` and `HistGradientBoostingClassifier`)
     - using `class_weights="balanced"` to handle the class imbalance
   - `scikit-learn` (ensemble) `VotingClassifier`
     - using `class_weights="balanced"` to handle the class imbalance
   - `xgboost`'s `XGBClassifier` using the `scikit-learn` API
     - using `scale_pos_weight`, determined from the combined train+validation data, to handle the class imbalance

As mentioned in the **About** section above, in each experiment, the choice of features and classifier will be changed in order to determine the best features. The pre-processing will change depending on the chosen features. `HistGradientBoostingClassifier` can internally handle categorical features, so it will be used with or without one-hot encoding.

Due to the class imbalance, we will determine the optimal decision threshold for a classifier. In order to optimize this threshold, we will use three custom methods defined in `src/cc_churn/tuning.py`

1. First, we will perform nested cross-validation by passing `scikit-learn`'s `TunedThresholdClassifierCV()` to `cross_validate()`. For a single ML experiment, `TunedThresholdClassifierCV()` performs internal cross-validation (CV) to find the optimal classifier decision threshold for the classifier step in the ML pipeline using stratified five-fold cross-validation and `prauc` as the evaluation metric. `cross_validate()` then uses outer cross-validation to evaluate the performance of this pipeline, for the selected features, with an optimal decision threshold again using stratified five-fold cross-validation.

   The outer CV returns a `DataFrame` with the train and test fold scores for all five outer CV folds (one row per fold) and for multiple evaluation metrics (one column per metric). The benefit of this approach is that inner CV optimizes the decision threshold while outer CV evaluates generalized model performance on data not used to tune the threshold. This approach avoids data leakage and gives an unbiased estimate of model performance. This is performed by the `tune_threshold_cv()` method.
2. Next, we append the best decision threshold from inner CV to each row of this `DataFrame`. Each row of the `DataFrame` corresponds to a different outer CV fold. For a given experiment, five-fold outer CV gives a `DataFrame` with five rows. However, the best decision threshold is determined separately for each outer CV fold. So, each outer fold sees a different training subset that has slightly different class balance, probability distributions and decision boundary behavior. With this in mind, the optimal threshold is data-dependent, and naturally varies across folds. For this reason, the best decision threshold differs from row to row (outer CV fold to fold) of this `DataFrame`.

   Although they are different, the best thresholds should be similar, but not identical, to each other across the five outer CV folds. A small variation in their values indicates the model is stable. On the other hand, a large variation suggests model instability. This step is implemented using the `combine_cv_scores_thresholds()` method.
3. In practice, we won't be using five different best thresholds to make the final predictions. Instead, as is done in cross-validation, we aggregate the evaluation scores and best thresholds across all five outer CV folds and get the average. So, this last aggregation step is performed using the `agg_cv_scores_thresholds()` method.

The three methods above are combined into a wrapper named `validate()` in `src/cc_churn/validation.py`, that calls them in succession.

(ml-experiments)=
### ML Experiments

The following five ML experiments are run

1. (experiment-1)=
   experiment 1
   - **features:** only select numerical features using
     - all uncorrelated features
     - the first from every pair of correlated features
     - `feat_group` = `[numerical_1]`
   - **pre-processing:**
     - min-max scaling for all numerical features
   - **classifiers:**
     - perform six runs within this experiment, with a different classifier used in each run
       - `LogisticRegression` (with `class_weights="balanced"`)
       - `HistGradientBoostingClassifier`
       - `RandomForestClassifier`
       - `XGBClassifier`
       - `LogisticRegression` (without `class_weights="balanced"`)
       - `VotingClassifier` (soft voting) that combines the first four classifiers from above
2. (experiment-2)=
   experiment 2
   - **features:** only select numerical features using
     - all uncorrelated features
     - the second from every pair of correlated features
     - `feat_group` = `[numerical_2]`
   - **pre-processing:**
     - same as in experiment 1
   - **classifiers:** same as in experiment 1
3. (experiment-3)=
   experiment 3
   - **features:** select numerical features from experiment 1 and ordinal features
     - `feat_group` = `[numerical_1,ordinal]`
   - **pre-processing:**
     - min-max scaling for all numerical features
     - ordinal encoding for ordinal features
   - **classifiers:** same as in experiments 1 and 2
4. (experiment-4)=
   experiment 4
   - **features:** select all features
     - numerical features from experiment 1
     - ordinal features from experiment 3
     - categorical features
     - `feat_group` = `[numerical_1,ordinal,categorical_ohe_encoding]`
   - **pre-processing:**
     - min-max scaling for all numerical features
     - ordinal encoding for ordinal features
     - one-hot encoding for categorical features
   - **classifiers:**
     - perform five runs within this experiment, with a different classifier used in each run
       - `LogisticRegression` (with `class_weights="balanced"`)
       - `RandomForestClassifier`
       - `XGBClassifier`
       - `LogisticRegression` (without `class_weights="balanced"`)
       - `VotingClassifier` (soft voting) that combines the first three classifiers from above
5. (experiment-5)=
   experiment 5
   - **features:** select all features
     - same as in experiment 4
     - `feat_group` = `[numerical_1,ordinal,categorical_no_encoding]`
   - **pre-processing**
     - min-max scaling for all numerical features
     - ordinal encoding for ordinal features
     - categorical features
       - cast as the `category` data type
       - no one-hot encoding for categorical features
   - **classifiers:**
     - perform one run within this experiment, with a classifier that is directed to internally encode features with the `category` dtype
       - `HistGradientBoostingClassifier`

Experiments 3, 4 and 5, which add categorical and ordinal features, will determine if sensitive features can be dropped without negatively impacting model performance.

```{important} ML Experiment versus Experiment Run
Metaflow considers each flow run to be an experiment run. A run ID is assigned to each run of a flow. No experiment ID is assigned. So, we will use the Metaflow run ID to identify each experiment run. For a specific experiment run, one combination of model, feature list and feature pre-processing is used. We will use the above experiment numbers to identify the experiments. So, each experiment can have multiple runs.
```

### ML Model Selection

Below is the list of classifiers to be validated for combinations of numerical and numerical+ordinal features in experiments 1, 2 and 3

In [ ]:
models = {
    "LogisticRegression_imbalanced": LogisticRegression(
        class_weight=None, random_state=42
    ),
    "LogisticRegression": LogisticRegression(
        class_weight="balanced", random_state=42
    ),
    "HistGradientBoostingClassifier": skens.HistGradientBoostingClassifier(
        max_depth=3,
        l2_regularization=0.25,
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForestClassifier": skens.RandomForestClassifier(
        n_estimators=800,
        max_depth=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "XGBClassifier": xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        gamma=20,
        max_depth=0,
        eta=0.05,
        min_child_weight=15,
        scale_pos_weight=5.223088923556943,
        random_state=42,
        n_jobs=-1,
    ),
    "Ensemble__VotingClassifier": skens.VotingClassifier(
        estimators=[
            (
                "lr",
                LogisticRegression(class_weight="balanced", random_state=42),
            ),
            (
                "hbc",
                skens.HistGradientBoostingClassifier(
                    max_depth=3,
                    l2_regularization=0.25,
                    class_weight="balanced",
                    random_state=42,
                ),
            ),
            (
                "rf",
                skens.RandomForestClassifier(
                    n_estimators=800,
                    max_depth=3,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
            (
                "xgb",
                xgb.XGBClassifier(
                    objective="binary:logistic",
                    eval_metric="logloss",
                    enable_categorical=True,
                    gamma=20,
                    max_depth=0,
                    eta=0.05,
                    min_child_weight=15,
                    scale_pos_weight=5.223088923556943,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ],
        voting="soft",
    ),
}

:::{hint} Specifying Class Weights for **`XGBClassifier`**

For `XGBClassifier()`, the value of the [`scale_pos_weight` parameter was determined during EDA](./03_eda_v2.ipynb#scale-pos-weight) and is hard-coded here to handle the class imbalance of the data.
:::

Below is the list of classifiers to be validated, for features that include categoricals with one-hot encoding in experiment 4

In [ ]:
models_ohe_cat_encoding = {
    "LogisticRegression_imbalanced": LogisticRegression(
        class_weight=None, random_state=42
    ),
    "LogisticRegression": LogisticRegression(
        class_weight="balanced", random_state=42
    ),
    "RandomForestClassifier": skens.RandomForestClassifier(
        n_estimators=800,
        max_depth=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "XGBClassifier": xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        gamma=20,
        max_depth=0,
        eta=0.05,
        min_child_weight=15,
        scale_pos_weight=5.223088923556943,
        random_state=42,
        n_jobs=-1,
    ),
    "Ensemble__VotingClassifier": skens.VotingClassifier(
        estimators=[
            (
                "lr",
                LogisticRegression(class_weight="balanced", random_state=42),
            ),
            (
                "rf",
                skens.RandomForestClassifier(
                    n_estimators=800,
                    max_depth=3,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
            (
                "xgb",
                xgb.XGBClassifier(
                    objective="binary:logistic",
                    eval_metric="logloss",
                    gamma=20,
                    max_depth=0,
                    eta=0.05,
                    min_child_weight=15,
                    scale_pos_weight=5.223088923556943,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ],
        voting="soft",
    ),
}

The classifiers to be validated for categorical features without any encoding, but used with features converted to the `category` dtype in experiment 5, are shown below

In [ ]:
models_no_cat_encoding = {
    "HistGradientBoostingClassifier": skens.HistGradientBoostingClassifier(
        max_depth=3,
        l2_regularization=0.25,
        categorical_features=[11, 12, 13],
        class_weight="balanced",
        random_state=42,
    ),
}

For passing different classifier objects to the same Metaflow flow, the models are now serialized to disk locally in `models/<model-name>.joblib` [using `joblib`](https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html) and their filepaths are stored as strings in a dictionary that can be use to parameterize a Metaflow flow using the `model_fpath` parameter

In [ ]:
models_fpaths = {"01": {}, "02": {}, "03": {}, "04": {}, "05": {}}
for expt_num, models_dict in zip(
    [1, 2, 3, 4, 5],
    [models, models, models, models_ohe_cat_encoding, models_no_cat_encoding],
):
    experiment_num_str = str(expt_num).zfill(2)
    for k, model in models_dict.items():
        model_fpath = models_dir / experiment_num_str / f"{k}.joblib"
        if not model_fpath.is_file():
            joblib.dump(model, model_fpath)
            print(f"Serialized {k} for experiment {experiment_num_str}")
        models_fpaths[experiment_num_str].update({k: str(model_fpath)})

### Experiments

The Metaflow flow for validation has the following three steps

1. `extract()`
   - load train+validation data from R2 bucket and separate features (`X`) from class labels (`y`)
2. `preprocess`
   - defines a list of transformers and preprocessors than can be passed to a `scikit-learn` `Pipeline` that
     - transforms categorical features by grouping infrequent categories, using a [custom `scikit-learn` transformer](https://scikit-learn.org/stable/modules/generated/sklearn.base.TransformerMixin.html#sklearn.base.TransformerMixin)
     - pre-processes numerical features [using `MinMaxScaler()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)
     - encodes ordinal features [using `OrdinalEncoder()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html)
       - this encoder assumes infrequent category grouping has been performed in the previous step of the pipeline
     - dummy encodes categorical features [using `OneHotEncoder()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) or does not encode them
3. `validate`
   - defines `scikit-learn` `Pipeline` with transformation, preprocessing and classification steps
   - implements nested cross-validation as discussed earlier and stores all outer [CV outputs from `cross_validate()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html) in a `DataFrame` in the class attribute `df_cv`

The code for the custom `scikit-learn` transformer to group infrequent categories in step 2. of the flow is shown below

In [ ]:
pygments_highlight(
    fpath=str(PROJ_ROOT / "src" / "cc_churn" / "transformers.py"),
    unwanted_lines=(
        list(range(0, 11))
        + list(range(14, 48))
        + list(range(49, 61))
        + list(range(65, 81))
        + list(range(98, 109))
    ),
    style="friendly",
)

(validation-flow)=
The Metaflow validation flow is defined below

In [ ]:
import json
import os
from pathlib import Path

import boto3
import joblib
import numpy as np
import pandas as pd
import r2.io_utils as r2io
import sklearn.model_selection as mds
import sklearn.preprocessing as pp
from cc_churn.scoring import get_scorers
from cc_churn.transformers import CategoryCombiner2
from metaflow import FlowSpec, NBRunner, Parameter, step
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


class ValidationFlow(FlowSpec):
    r2_keys = Parameter(
        name="r2_keys", help="R2 keys", default="{'train': 'C', 'val': 'D'}"
    )
    dtypes_ordinals = Parameter(
        name="dtypes_ordinals",
        help="ordinal datatypes",
        default="{'B': 'string[pyarrow]'}",
    )
    dtypes_categoricals = Parameter(
        name="dtypes_categoricals",
        help="categorical datatypes",
        default="{'G': 'string[pyarrow]'}",
    )
    numericals = Parameter(
        name="numericals", help="numerical features", default="['A', 'B']"
    )
    ordinals = Parameter(
        name="ordinals", help="ordinal features", default="['P', 'A']"
    )
    categoricals = Parameter(
        name="categoricals", help="categorical features", default="['T', 'B']"
    )
    feat_group = Parameter(
        name="feat_group", help="feature set", default="numericals_1"
    )
    model_fpath = Parameter(
        name="model_fpath", help="model filepath", default="model.joblib"
    )

    @step
    def start(self):
        self.primary_metric_threshold_optim = "f2"
        self.metrics_list_val = ["prauc", "f2", "recall", "rocauc"]
        self.cv = mds.StratifiedKFold(5, shuffle=True, random_state=88)
        self.next(self.extract)

    @step
    def extract(self):
        s3_client = boto3.client(
            "s3",
            endpoint_url=(
                f"https://{os.getenv('ACCOUNT_ID')}.r2.cloudflarestorage.com"
            ),
            aws_access_key_id=os.getenv("ACCESS_KEY_ID_USER2"),
            aws_secret_access_key=os.getenv("SECRET_ACCESS_KEY_USER2"),
            region_name="auto",
        )
        df_train, df_val = [
            r2io.pandas_read_parquet_r2(
                s3_client,
                os.getenv("BUCKET_NAME"),
                json.loads(self.r2_keys)[split_type],
            )
            .astype(json.loads(self.dtypes_ordinals))
            .astype(json.loads(self.dtypes_categoricals))
            for split_type in ["train", "val"]
        ]
        df = pd.concat([df_train, df_val], ignore_index=True)

        self.X = df.drop(columns=["is_churned"])
        self.y = df["is_churned"]
        self.next(self.preprocess)

    @step
    def preprocess(self):
        numeric_transformer = Pipeline([("scaler", pp.MinMaxScaler())])
        categorical_transformer = Pipeline(
            [("ohe", pp.OneHotEncoder(drop="if_binary", sparse_output=False))]
        )
        ordinal_transformer = Pipeline(
            [
                (
                    "oe",
                    pp.OrdinalEncoder(
                        categories=[
                            [
                                "Unknown",
                                "Less than $40K",
                                "$40K - $60K",
                                "$60K - $80K",
                                "$80K+",
                            ],
                            [
                                "Unknown",
                                "Uneducated",
                                "High School",
                                "College",
                                "Graduate",
                                "Post-Graduate",
                            ],
                        ],
                        handle_unknown="use_encoded_value",
                        unknown_value=np.nan,
                    ),
                )
            ]
        )

        feat_group_dict = {
            "[numericals_1]": [
                ("num", numeric_transformer, json.loads(self.numericals))
            ],
            "[numericals_2]": [
                ("num", numeric_transformer, json.loads(self.numericals))
            ],
            "[numericals_1,ordinals]": [
                ("num", numeric_transformer, json.loads(self.numericals)),
                ("ord", ordinal_transformer, json.loads(self.ordinals)),
            ],
            "[numericals_1,ordinals,categoricals_ohe_encoding]": [
                ("num", numeric_transformer, json.loads(self.numericals)),
                ("ord", ordinal_transformer, json.loads(self.ordinals)),
                (
                    "cat",
                    categorical_transformer,
                    json.loads(self.categoricals)
                ),
            ],
            "[numericals_1,ordinals,categoricals_no_encoding]": [
                ("num", numeric_transformer, json.loads(self.numericals)),
                ("ord", ordinal_transformer, json.loads(self.ordinals)),
                ("cat", "passthrough", json.loads(self.categoricals)),
            ],
        }
        transformers = feat_group_dict[self.feat_group]

        preprocessor = ColumnTransformer(
            transformers, remainder="drop", verbose_feature_names_out=False
        ).set_output(transform="pandas")

        cat_ord_grouper = CategoryCombiner2().set_output(transform="pandas")
        self.transformers_preprocessors = [
            ("catgroup", cat_ord_grouper),
            ("pre", preprocessor),
        ]
        self.next(self.validate)

    @step
    def validate(self):
        scorers = get_scorers(self.metrics_list_val)
        clf = joblib.load(self.model_fpath)

        self.pipe = Pipeline(
            self.transformers_preprocessors + [("clf", clf)]
        ).set_output(transform="pandas")

        tuned_model = mds.TunedThresholdClassifierCV(
            estimator=self.pipe,
            scoring=scorers[self.primary_metric_threshold_optim],
            cv=self.cv,
        )
        self.df_cv = pd.DataFrame(
            mds.cross_validate(
                tuned_model,
                self.X,
                self.y.to_numpy().ravel(),
                scoring=scorers,
                cv=self.cv,
                params=None,
                return_train_score=True,
                return_estimator=True,
                n_jobs=-1,
            )
        ).assign(
            feat_group=self.feat_group,
            model_name=Path(self.model_fpath).name.replace(".joblib", ""),
        )
        self.next(self.fit)

    @step
    def fit(self):
        _ = self.pipe.fit(self.X, self.y)
        self.next(self.end)

    @step
    def end(self):
        pass


_ = NBRunner(ValidationFlow, pylint=False).nbrun(
    r2_keys=json.dumps({"train": r2_key_train, "val": r2_key_val}),
    dtypes_ordinals=json.dumps(dtypes_ordinals),
    dtypes_categoricals=json.dumps(dtypes_categoricals),
    numericals=json.dumps(
        numeric_features_1
        if "numericals_1" in feat_group
        else numeric_features_2
    ),
    ordinals=json.dumps(ordinal_features),
    categoricals=json.dumps(categorical_features),
    feat_group=feat_group,
    model_fpath=models_fpaths[str(experiment_num).zfill(2)][clf_name],
)

```{note}
In the `preprocess()` step of this flow, a dictionary `feat_group_dict` is defined that contains the list of transformers and corresponding features for the feature grouping used in all five experiments. This dictionary is sliced using an instance variable `.feat_group` that stores the user's input for a single feature group. The resulting list of transformers is stored in `transformers`. This is list is finally used to assemble the overall list of transformers and feature pre-processors (`transformers_preprocessors`) that goes directly into a `scikit-learn.Pipeline` from the instance variable `.pipe` in the `validate()` step of the flow.

In the `validate()` step, a dictionary of scorers is loaded based on a list that was hard-coded in the `start()` step. This dictionary is then sliced to get the primary scoring metric for the inner (`f2`) and outer (`prauc`) CV of nested CV [for use in threshold tuning and model selection](../references/scope/06_analysis.md#choice-of-metrics) respectively. The ML model is then loaded from the flow parameter `model_path`. Finally, nested CV is run and the results are stored with experiment and run metadata (`feat_group` and `model_name`) in a `DataFrame` named `df_cv`. Since all [instance variables are persisted to disk](https://docs.metaflow.org/internals/technical-overview#step-code), `df_cv` is not exported to the R2 bucket.
```

```{attention}
The `start()` step only performs lightweight data initialization and metadata setup.
```

(train-all-validation)=
```{important}
The `fit()` step simply trains the ML model on all available data (combined train and validation split). This way the trained model for a specific experiment run, that is trained on all available data, can be directly used to make predictions without training.
```

[As discussed earlier](#ml-experiments), the above metaflow `Flow` can be run with the following combinations of the `feat_group` and `model_fpath` parameters

1. [**experiment 1**](#experiment-1)
   - numerical features with the models defined in `models`
   - ```python
     feat_group = "[numericals_1]"
     model_fpath = models_fpaths["01"]["HistGradientBoostingClassifier"]
     model_fpath = models_fpaths["01"]["LogisticRegression_imbalanced"]
     model_fpath = models_fpaths["01"]["LogisticRegression"]
     model_fpath = models_fpaths["01"]["RandomForestClassifier"]
     model_fpath = models_fpaths["01"]["XGBClassifier"]
     model_fpath = models_fpaths["01"]["Ensemble__VotingClassifier"]
     ```
2. [**experiment 2**](#experiment-2)
   - numerical features with the models defined in `models`
   - ```python
     feat_group = "[numericals_2]"
     model_fpath = models_fpaths["02"]["HistGradientBoostingClassifier"]
     model_fpath = models_fpaths["02"]["LogisticRegression_imbalanced"]
     model_fpath = models_fpaths["02"]["LogisticRegression"]
     model_fpath = models_fpaths["02"]["RandomForestClassifier"]
     model_fpath = models_fpaths["02"]["XGBClassifier"]
     model_fpath = models_fpaths["02"]["Ensemble__VotingClassifier"]
     ```
3. [**experiment 3**](#experiment-3)
   - numerical and ordinal features with the models defined in `models`
   - ```python
     feat_group = "[numericals_1,ordinals]"
     model_fpath = models_fpaths["03"]["HistGradientBoostingClassifier"]
     model_fpath = models_fpaths["03"]["LogisticRegression_imbalanced"]
     model_fpath = models_fpaths["03"]["LogisticRegression"]
     model_fpath = models_fpaths["03"]["RandomForestClassifier"]
     model_fpath = models_fpaths["03"]["XGBClassifier"]
     model_fpath = models_fpaths["03"]["Ensemble__VotingClassifier"]
     ```
4. [**experiment 4**](#experiment-4)
   - numerical, ordinal and dummy-encoded categorical features with the models defined in `models_ohe_cat_encoding`
   - ```python
     feat_group = "[numericals_1,ordinals,categoricals_ohe_encoding]"
     model_fpath = models_fpaths["04"]["LogisticRegression_imbalanced"]
     model_fpath = models_fpaths["04"]["LogisticRegression"]
     model_fpath = models_fpaths["04"]["RandomForestClassifier"]
     model_fpath = models_fpaths["04"]["XGBClassifier"]
     model_fpath = models_fpaths["04"]["Ensemble__VotingClassifier"]
     ```
5. [**experiment 5**](#experiment-5)
   - numerical, ordinal and unencoded categorical features with the model defined in `models_no_cat_encoding`
   - ```python
     feat_group = "[numericals_1,ordinals,categoricals_no_encoding]"
     model_fpath = models_fpaths["05"]["HistGradientBoostingClassifier"]
     ```

## Summary

We have defined a parameterized Metaflow flow to run ML experiments as part of the validation process. This flow can be run with parameters specific to each experiment as described in the [experiments listed above](#ml-experiments).

In the next step, we will inspect the outputs of all runs of this flow.